In [2]:
# @title Version Control with Github

# --- Prerequisites ---
# 1. Add SSH Private Key: Secret named "github_colab_keys"
# 2. Add GitHub PAT: Secret named "github_longhaiSK_token" (Fine-grained: All Repos + Metadata:Read-only).
# ---------------------

# Install necessary libraries
!pip install ipywidgets --quiet

# Import libraries
import os
import subprocess
import shlex
import json
from google.colab import userdata
from IPython.display import display, clear_output
import ipywidgets as widgets
import shutil

# --- Configuration ---
GITHUB_USERNAME = "longhaiSK" # Your GitHub username
SSH_SECRET_KEY_NAME = "github_colab_keys" # Name of the SSH key secret
PAT_SECRET_NAME = "github_longhaiSK_token" # Name of the PAT secret
SSH_DIR = os.path.expanduser("~/.ssh")
SSH_KEY_PATH = os.path.join(SSH_DIR, "id_rsa_colab_widget")
KNOWN_HOSTS_PATH = os.path.join(SSH_DIR, "known_hosts")
REPO_CLONE_DIR_BASE = f"/content/{GITHUB_USERNAME}" # Base directory for clones

# --- Global flags and Log List ---
ssh_setup_ok = False; install_gh_ok = False; gh_auth_ok = False; repo_fetch_ok = False
pat_token_content = None; setup_log_messages = []

# --- 1. Set up SSH Key & Git Config ---
setup_log_messages.append("Setting up SSH Key & Git Config...")
try:
    ssh_key_content = userdata.get(SSH_SECRET_KEY_NAME); os.makedirs(SSH_DIR, mode=0o700, exist_ok=True)
    normalized_ssh_key = ssh_key_content.replace('\r\n', '\n').rstrip('\n') + '\n'
    with open(SSH_KEY_PATH, "w") as f: f.write(normalized_ssh_key)
    os.chmod(SSH_KEY_PATH, 0o600); setup_log_messages.append(f"-> SSH key saved to {SSH_KEY_PATH}.")
    try:
        keyscan_output = subprocess.check_output(['ssh-keyscan', '-t', 'rsa', 'github.com'], text=True, stderr=subprocess.DEVNULL)
        known_hosts_content = "";
        if os.path.exists(KNOWN_HOSTS_PATH):
             with open(KNOWN_HOSTS_PATH, "r") as f_read: known_hosts_content = f_read.read()
        if 'github.com' not in known_hosts_content:
            with open(KNOWN_HOSTS_PATH, "a") as f_known_hosts: f_known_hosts.write(keyscan_output)
            setup_log_messages.append("-> Added github.com fingerprint.")
        else: setup_log_messages.append("-> github.com already in known_hosts.")
    except Exception as e: setup_log_messages.append(f"-> Warn: ssh-keyscan failed: {e}.")
    setup_log_messages.append("-> Configuring Git user identity...")
    try:
        user_email = "longhaiSK@example.com" # <-- Replace with your actual email
        subprocess.run(['git', 'config', '--global', 'user.name', GITHUB_USERNAME], check=True, capture_output=True)
        subprocess.run(['git', 'config', '--global', 'user.email', user_email], check=True, capture_output=True)
        setup_log_messages.append(f"-> Git identity set: {GITHUB_USERNAME} <{user_email}>.")
    except Exception as git_config_e: setup_log_messages.append(f"-> WARN: Git config failed: {git_config_e}")
    git_ssh_command = f"ssh -i {SSH_KEY_PATH} -o UserKnownHostsFile={KNOWN_HOSTS_PATH} -o StrictHostKeyChecking=no"
    os.environ['GIT_SSH_COMMAND'] = git_ssh_command; setup_log_messages.append("-> GIT_SSH_COMMAND environment variable set.")
    ssh_setup_ok = True
except Exception as e: setup_log_messages.append(f"ERROR during SSH setup: {e}"); print(f"ERROR during SSH setup: {e}")

# --- 2. Install gh CLI ---
setup_log_messages.append("\nChecking for GitHub CLI (gh)...")
install_gh_ok = False
try:
    if shutil.which('gh'):
        gh_version_result = subprocess.run(['gh', '--version'], capture_output=True, text=True)
        setup_log_messages.append(f"-> GitHub CLI already installed ({gh_version_result.stdout.splitlines()[0].strip()}).")
        install_gh_ok = True
    else:
        setup_log_messages.append("-> GitHub CLI not found. Attempting installation..."); subprocess.run("type -p curl >/dev/null || (apt-get update -qq && apt-get install curl -y -qq --no-install-recommends)", shell=True, check=True, capture_output=True)
        subprocess.run("curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg | dd of=/usr/share/keyrings/githubcli-archive-keyring.gpg >/dev/null 2>&1", shell=True, check=True, capture_output=True); subprocess.run("chmod go+r /usr/share/keyrings/githubcli-archive-keyring.gpg", shell=True, check=True, capture_output=True)
        subprocess.run('echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" | tee /etc/apt/sources.list.d/github-cli.list > /dev/null', shell=True, check=True, capture_output=True); setup_log_messages.append("-> Added gh repository source.")
        subprocess.run("apt-get update -qq", shell=True, check=True, capture_output=True); setup_log_messages.append("-> Running apt-get install gh...")
        subprocess.run("apt-get install gh -y -qq --no-install-recommends", shell=True, check=True, capture_output=True)
        if shutil.which('gh'):
             gh_version_result = subprocess.run(['gh', '--version'], capture_output=True, text=True); setup_log_messages.append(f"-> GitHub CLI installed successfully ({gh_version_result.stdout.splitlines()[0]})."); install_gh_ok = True
        else: raise Exception("gh command still not found after installation attempt.")
except Exception as e: error_msg = f"ERROR during gh check/installation: {e}"; setup_log_messages.append(error_msg); print(error_msg); install_gh_ok = False

# --- 3. Authenticate gh CLI using PAT ---
setup_log_messages.append(f"\nAuthenticating gh CLI using secret '{PAT_SECRET_NAME}'...")
if install_gh_ok:
    try:
        pat_token_content = userdata.get(PAT_SECRET_NAME)
        if not pat_token_content: raise ValueError(f"PAT Secret '{PAT_SECRET_NAME}' not found or empty.")
        os.environ['GH_TOKEN'] = pat_token_content; setup_log_messages.append("-> GH_TOKEN env var set.")
        result = subprocess.run(['gh', 'auth', 'status'], capture_output=True, text=True, env=os.environ)
        if result.returncode == 0 and "Logged in" in result.stdout: setup_log_messages.append("-> gh auth successful."); gh_auth_ok = True
        else: raise Exception(f"gh auth status failed.\nStdout: {result.stdout}\nStderr: {result.stderr}")
    except Exception as e: error_msg = f"ERROR during gh authentication: {e}"; setup_log_messages.append(error_msg); print(error_msg)
else: setup_log_messages.append("-> Skipping gh auth: installation failed.")

# --- 4. Fetch GitHub Repositories (using gh CLI) ---
repo_options = {"-- Select a Repository --": None}; default_branches = {}
if gh_auth_ok:
    setup_log_messages.append(f"\nFetching repositories for '{GITHUB_USERNAME}' via gh CLI...")
    try:
        cmd = ['gh', 'repo', 'list', GITHUB_USERNAME, '--limit', '500', '--json', 'name,sshUrl,defaultBranchRef']
        result = subprocess.run(cmd, capture_output=True, text=True, check=True, env=os.environ); repos_data = json.loads(result.stdout)
        count = 0; temp_repo_options = {}
        for repo in repos_data:
            repo_name = repo.get('name'); ssh_url = repo.get('sshUrl'); default_branch_info = repo.get('defaultBranchRef'); default_branch = default_branch_info.get('name') if default_branch_info else 'main'
            if repo_name and ssh_url: temp_repo_options[repo_name] = ssh_url; default_branches[ssh_url] = default_branch; count += 1
        if count > 0: setup_log_messages.append(f"-> Fetched {count} repositories."); repo_options.update(sorted(temp_repo_options.items())); repo_fetch_ok = True
        else: setup_log_messages.append(f"-> No repositories found for '{GITHUB_USERNAME}'."); repo_options = {"-- No Repos --": None}
    except Exception as e: error_msg = f"ERROR fetching repos: {e}"; setup_log_messages.append(error_msg); print(error_msg); repo_options = {"-- Fetch Error --": None}
else: setup_log_messages.append("-> Skipping repo fetch: gh auth failed."); repo_options = {"-- Auth Error --": None}


# --- Widget Definitions ---
output_area = widgets.Textarea( value='', placeholder='Status messages and command output appear here...', description='', layout=widgets.Layout(height='250px', width='auto'), disabled=True )
repo_selector = widgets.Dropdown( options=repo_options, description='Repo:', style={'description_width': 'initial'}, disabled= not ssh_setup_ok or not gh_auth_ok or not repo_fetch_ok )
clone_button = widgets.Button(description="Clone", button_style='info', disabled=True, icon='download', tooltip='Clone selected repository if not present locally')
commit_message = widgets.Text( value='Update from Colab', placeholder='Enter commit message', description='Commit Msg:', style={'description_width': 'initial'}, disabled=True)
refresh_button = widgets.Button(description="Refresh File Status", button_style='info', icon='refresh', disabled=True, tooltip='Check for local changes')
changes_area = widgets.VBox([], layout=widgets.Layout(max_height='150px', overflow_y='auto', border='1px solid #ccc', padding='5px', margin='5px 0 0 0'))
commit_button = widgets.Button(description="Commit Selected", button_style='warning', icon='upload', disabled=True, tooltip='Stage and commit ONLY selected files')
push_button = widgets.Button(description="Push", button_style='success', icon='cloud-upload', disabled=True, tooltip='Push committed changes')
pull_button = widgets.Button(description="Pull", button_style='primary', icon='cloud-download', disabled=True, tooltip='Pull latest changes from remote')

# --- State Variables ---
current_repo_local_path = None; current_repo_ssh_url = None; current_default_branch = None

# --- Helper Functions ---
def run_git_command(command, working_dir):
    if not working_dir or not os.path.isdir(working_dir): output_area.value = f"Error: Invalid repo path '{working_dir}'"; return False
    return run_command(['git'] + command, working_dir=working_dir)

def run_command(command, working_dir=None):
    command_str = ""; success = False; stdout = ""; stderr = ""
    try: command_str = shlex.join(command)
    except AttributeError: command_str = ' '.join(shlex.quote(str(part)) for part in command)
    output_lines = [f"Running in {working_dir or '/content'}: {command_str}", "-" * 20]
    try:
        env = os.environ.copy(); result = subprocess.run(command, cwd=working_dir, capture_output=True, text=True, check=True, env=env)
        stdout = result.stdout; stderr = result.stderr; output_lines.append("Status: Success"); success = True
    except subprocess.CalledProcessError as e:
        stdout = e.stdout; stderr = e.stderr; output_lines.append(f"Status: Failed (Exit Code: {e.returncode})"); success = False
    except Exception as e: output_lines.append(f"Status: Failed (Python Exception)\nError: {e}"); success = False
    if stdout: output_lines.append("Output:\n" + stdout)
    if stderr: output_lines.append("Stderr:\n" + stderr)
    output_area.value = "\n".join(output_lines).strip(); return success

# --- >>> MODIFIED Function: Uses strip().split() <<< ---
def update_changes_display():
    """Gets git status, puts status in checkbox description."""
    global current_repo_local_path
    if not current_repo_local_path or not os.path.isdir(current_repo_local_path):
        changes_area.children = [widgets.Label("No repository selected or found.")]; commit_button.disabled = True; return

    new_widgets = []; commit_button.disabled = True
    try:
        cmd = ['git', 'status', '--porcelain']
        result = subprocess.run(cmd, cwd=current_repo_local_path, capture_output=True, text=True, check=False, env=os.environ)
        if result.returncode != 0 and result.stderr and not result.stdout:
             new_widgets.append(widgets.Label(f"Error getting git status: {result.stderr}"))
        elif not result.stdout.strip(): new_widgets.append(widgets.Label("No changes detected."))
        else:
            lines = result.stdout.strip().split('\n'); any_changes = False
            for line in lines:
                if not line: continue
                # --- >>> CORRECTED PARSING: strip() then split() <<< ---
                parts = line.strip().split(' ', 1)
                # --- >>> END CORRECTION <<< ---
                if len(parts) == 2:
                    status_code = parts[0] # Status XY (or ??) - No strip needed now
                    file_path = parts[1]   # Filename - No strip needed now unless filename has own spaces
                    # Handle potentially quoted paths (if filename itself has spaces etc)
                    if len(file_path) > 1 and file_path.startswith('"') and file_path.endswith('"'):
                         try: file_path = bytes(file_path[1:-1], 'ascii').decode('unicode_escape')
                         except Exception: file_path = file_path[1:-1].replace('\\"', '"').replace('\\\\', '\\')
                    any_changes = True
                    description_str = f"{file_path} ({status_code})" # Create description: "filename (XY)"
                    cb = widgets.Checkbox(value=True, description=description_str, indent=False, layout=widgets.Layout(width='auto'))
                    cb.file_path = file_path # Store the original path WITHOUT status
                    cb.status_code = status_code
                    new_widgets.append(cb) # Add checkbox directly
                else: new_widgets.append(widgets.Label(f"Unparsed: {line}"))
            commit_button.disabled = not any_changes
    except Exception as e: new_widgets.append(widgets.Label(f"Error processing git status: {e}"))
    changes_area.children = tuple(new_widgets)
# --- >>> END MODIFIED Function <<< ---

# --- Widget Callbacks ---
def on_repo_selection_change(change):
    global current_repo_local_path, current_repo_ssh_url, current_default_branch
    selected_ssh_url = change['new']; current_repo_ssh_url = selected_ssh_url; current_repo_local_path = None
    repo_name = None; clone_dest = None; output_lines = []
    clone_button.disabled = True; refresh_button.disabled = True; commit_message.disabled = True; commit_button.disabled = True; push_button.disabled = True; pull_button.disabled = True
    changes_area.children = []
    if selected_ssh_url:
        repo_name = next((name for name, url in repo_options.items() if url == selected_ssh_url), None)
        if repo_name:
            current_default_branch = default_branches.get(selected_ssh_url, 'main')
            clone_dest = os.path.join(REPO_CLONE_DIR_BASE, repo_name)
            if os.path.isdir(clone_dest): # Repo exists locally
                current_repo_local_path = clone_dest
                output_lines.append(f"Selected: {repo_name} (local exists)"); output_lines.append(f"Path: {clone_dest}"); output_lines.append("Use 'Pull' or 'Refresh'.")
                commit_message.disabled = False; push_button.disabled = False; pull_button.disabled = False; refresh_button.disabled = False
                update_changes_display()
            else: # Repo does not exist locally
                output_lines.append(f"Selected: {repo_name}"); output_lines.append(f"Ready to clone into: {clone_dest}")
                clone_button.disabled = False
        else: output_lines.append("Error: Could not map SSH URL to repo name.")
    elif repo_fetch_ok: output_lines.append("Select a repository from the dropdown.")
    else: output_lines.append("Setup failed or no repositories fetched.")
    output_area.value = "\n".join(output_lines)
repo_selector.observe(on_repo_selection_change, names='value')

def on_clone_button_clicked(b):
    global current_repo_local_path
    if not current_repo_ssh_url: output_area.value = "Error: No repository selected."; return
    repo_name = next((name for name, url in repo_options.items() if url == current_repo_ssh_url), None)
    if not repo_name: output_area.value = "Error: Could not determine repository name."; return
    clone_dest = os.path.join(REPO_CLONE_DIR_BASE, repo_name); current_repo_local_path = clone_dest
    os.makedirs(REPO_CLONE_DIR_BASE, exist_ok=True)
    output_area.value = f"Attempting to clone {repo_name}..."
    command = ['git', 'clone', current_repo_ssh_url, clone_dest]
    success = run_command(command)
    if success:
        commit_message.disabled = False; push_button.disabled = False; pull_button.disabled = False; refresh_button.disabled = False
        clone_button.disabled = True; update_changes_display()
    else:
        current_repo_local_path = None; clone_button.disabled = False; changes_area.children = []
        commit_message.disabled = True; commit_button.disabled = True; push_button.disabled = True; pull_button.disabled = True; refresh_button.disabled = True
clone_button.on_click(on_clone_button_clicked)

def on_refresh_button_clicked(b):
    if current_repo_local_path:
        output_area.value = f"Refreshing file status for {os.path.basename(current_repo_local_path)}..."
        update_changes_display()
        output_area.value = f"File status updated for {os.path.basename(current_repo_local_path)}.\nCheck/uncheck files to include in commit."
    else: output_area.value = "Select a repository first."
refresh_button.on_click(on_refresh_button_clicked)

# --- >>> MODIFIED: Commit logic iterates Checkboxes directly <<< ---
def on_commit_button_clicked(b):
    if not current_repo_local_path or not os.path.isdir(current_repo_local_path): output_area.value = f"Error: Repository directory not found: {current_repo_local_path}"; return
    msg = commit_message.value.strip();
    if not msg: output_area.value = "Error: Commit message cannot be empty."; return
    selected_files_to_stage = []
    # Iterate directly through Checkbox widgets in the changes_area
    for item_widget in changes_area.children:
        if isinstance(item_widget, widgets.Checkbox): # Process only checkboxes
            checkbox = item_widget
            if checkbox.value is True and hasattr(checkbox, 'file_path'):
                selected_files_to_stage.append(checkbox.file_path) # Use stored clean path

    if not selected_files_to_stage: output_area.value = "No files selected to commit. Check boxes then click Commit Selected."; return
    output_area.value = f"Staging {len(selected_files_to_stage)} selected file(s)..."
    add_success = run_git_command(['add'] + selected_files_to_stage, working_dir=current_repo_local_path)
    if not add_success: output_area.value += "\nError during 'git add'. Commit aborted."; return
    output_area.value += f"\nStaged {len(selected_files_to_stage)} files. Attempting commit..."
    commit_success = run_git_command(['commit', '-m', msg], working_dir=current_repo_local_path)
    if commit_success:
        output_area.value += "\nCommit successful! Refreshing status..."
        update_changes_display(); output_area.value += "\nFile status refreshed."
    else: output_area.value += "\nCommit failed. Check output above."
commit_button.on_click(on_commit_button_clicked)

# --- Push/Pull Callbacks ---
def on_push_button_clicked(b):
    if not current_repo_local_path or not os.path.isdir(current_repo_local_path): output_area.value = f"Error: Repository directory not found: {current_repo_local_path}"; return
    if not current_default_branch: output_area.value = "Error: Default branch unknown. Cannot push."; return
    run_git_command(['push', 'origin', current_default_branch], working_dir=current_repo_local_path)
push_button.on_click(on_push_button_clicked)

def on_pull_button_clicked(b):
    if not current_repo_local_path or not os.path.isdir(current_repo_local_path): output_area.value = f"Error: Repository directory not found: {current_repo_local_path}"; return
    if not current_default_branch: output_area.value = "Error: Default branch unknown. Cannot pull."; return
    success = run_git_command(['pull', 'origin', current_default_branch], working_dir=current_repo_local_path)
    if success: update_changes_display(); output_area.value += "\nFile status refreshed after pull."
pull_button.on_click(on_pull_button_clicked)

# --- Display Widgets ---
spacer = widgets.HTML(value="<div style='margin-top: 8px;'></div>")
controls = widgets.VBox([
    widgets.HBox([repo_selector, clone_button, pull_button]), spacer,
    widgets.HBox([refresh_button], layout=widgets.Layout(justify_content='flex-start')), changes_area, spacer,
    commit_message, spacer,
    widgets.HBox([commit_button, push_button]),
    widgets.HTML(value="<div style='margin-top: 8px;'>Git Command Outputs:</div>")
    ,
    output_area
])

# --- Set Initial Status Message in Textarea ---
initial_message = ""
if not ssh_setup_ok or not gh_auth_ok or not repo_fetch_ok: initial_message = "\n\nSetup failed or no repositories found/fetched. Widgets may be disabled.\nPlease check details in the log above."
elif repo_selector.value is None: initial_message = "\n\nSetup complete. Select a repository from the dropdown to begin."
output_area.value = "\n".join(setup_log_messages) + initial_message

# --- Display ---

display(controls)

# --- Trigger initial repo check if applicable ---
if ssh_setup_ok and gh_auth_ok and repo_fetch_ok and repo_selector.value is not None:
     on_repo_selection_change({'new': repo_selector.value, 'owner': repo_selector, 'old': None})

ModuleNotFoundError: No module named 'google.colab'

In [3]:
# @title Install rpy2 and Mount Google Drive

!pip install rpy2

%load_ext rpy2.ipython

  Installing build dependencies ...done
  Getting requirements to build wheel done
  Preparing metadata (pyproject.toml) ...done
  Created wheel for rpy2: filename=rpy2-3.5.17-cp310-cp310-macosx_10_15_x86_64.whl size=256013 sha256=f805ae33b4e8d6d0c87f75d32abb3b5937bb7c3b8e0df926fbeba81d1b7df432
  Stored in directory: /Users/lol553/Library/Caches/pip/wheels/a0/a2/a6/9a390508fa6c600d5363cea2e146f45fb4feffa4d2ec47ba93
Successfully built rpy2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
# @title Testing R runrime
%%R
print(R.version.string)
print("Hello in R")

UsageError: Line magic function `%%R` not found.


In [5]:
# @title Install Shinylive Packages to Google Drive

%load_ext rpy2.ipython

%%R

# --- 1. Mount Google Drive ---
# Ensure your Google Drive is mounted. Typically done via Colab interface.
# Example command (might need user interaction in Colab):
# drive.mount('/content/drive')
# Make sure this step is completed before running the rest.
print("Ensure Google Drive is mounted at /content/drive")

# --- 2. Define Persistent Library Path on Google Drive ---
# Get the major.minor R version string (e.g., "4.3")
r_version_string <- paste(R.version$major, R.version$minor, sep=".")
r_version_folder <- paste0("R_", r_version_string) # Creates folder like R_4.3

# Define the base path *on your Google Drive* for R packages
# CHANGE 'MyDrive' if your main folder has a different name
drive_r_libs_base <- "/content/drive/MyDrive/R_Packages_Colab"
persistent_lib_path <- file.path(drive_r_libs_base, r_version_folder)

# Create the directory on Drive if it doesn't exist
if (!dir.exists(persistent_lib_path)) {
  # Create directories recursively, suppress warnings if it already exists mid-creation
  dir.create(persistent_lib_path, recursive = TRUE, showWarnings = FALSE)
  # Check again after creation attempt
  if(dir.exists(persistent_lib_path)){
      print(paste("Created library directory:", persistent_lib_path))
  } else {
      stop(paste("Failed to create library directory:", persistent_lib_path, "- Check Drive permissions and path."))
  }
} else {
  print(paste("Using existing library directory:", persistent_lib_path))
}

# --- 3. Set .libPaths() ---
# Add the persistent path to the *beginning* of the library search paths
# This ensures R looks here FIRST.
# Remove any potential duplicates before adding
current_libs <- .libPaths()
current_libs <- current_libs[current_libs != persistent_lib_path] # Remove if already exists
.libPaths(c(persistent_lib_path, current_libs))

print("Current .libPaths():")
print(.libPaths())


# --- 4. Install Packages Intelligently ---

# Function to check availability and location
# Returns TRUE if the package is loadable AND located in the persistent library
# Returns FALSE otherwise
check_package_persistent <- function(pkg_name, persistent_path) {
  pkg_available <- requireNamespace(pkg_name, quietly = TRUE)
  if (!pkg_available) {
    return(FALSE) # Not loadable from anywhere
  }

  # If loadable, check its location
  pkg_location <- ""
  tryCatch({
    pkg_location <- find.package(pkg_name)
    # Use normalizePath for robust comparison
    is_in_persistent <- startsWith(normalizePath(pkg_location, mustWork = FALSE), normalizePath(persistent_path, mustWork = FALSE))
    return(is_in_persistent)
  }, error = function(e) {
    # If find.package fails for a loaded package (unlikely but possible), treat as not verified
    # print(paste("Warning: Could not determine location for seemingly available package", pkg_name, ":", e$message))
    return(FALSE)
  })
}

# --- Install Standard Packages ---
required_packages <- c("pak", "shiny", "httpuv") # Install pak first if using it for others

print("--- Checking and Installing Standard Packages ---")
for (pkg in required_packages) {
  cat("\nChecking:", pkg, "...\n")
  if (check_package_persistent(pkg, persistent_lib_path)) {
    cat(pkg, "is already available in the persistent library:", find.package(pkg), "\n")
  } else {
    # Reason for installation
    if (!requireNamespace(pkg, quietly = TRUE)) {
       cat(pkg, "is not available. Installing to", persistent_lib_path, "...\n")
    } else {
       cat(pkg, "is available but not in the persistent library (found at:", find.package(pkg), ").\n")
       cat("Re-installing", pkg, "to persistent library:", persistent_lib_path, "...\n")
       # Optional: You could remove the old one first if it causes issues, but usually not needed.
       # remove.packages(pkg, lib = find.package(pkg))
    }

    # Install the package to the persistent library
    install.packages(pkg, lib = persistent_lib_path, repos = "https://cloud.r-project.org/", quiet = FALSE) # Set quiet=FALSE for visibility

    # Verify installation attempt
    Sys.sleep(1) # Brief pause to allow filesystem changes to register
    if (check_package_persistent(pkg, persistent_lib_path)) {
      cat("Successfully installed and verified", pkg, "in", persistent_lib_path, "\n")
    } else {
      warning(paste("Failed to install or verify", pkg, "in", persistent_lib_path))
      # Check if it installed but isn't loadable
      if(pkg %in% installed.packages(lib.loc = persistent_lib_path)[,"Package"] && !requireNamespace(pkg, lib.loc=persistent_lib_path, quietly = TRUE)){
          cat("Package", pkg, "files seem present but package is not loadable. Check for R version compatibility or installation errors.\n")
      }
    }
  }
}

# --- Install shinylive using pak ---
shinylive_pkg <- "shinylive"
shinylive_repo <- "posit-dev/r-shinylive"

print(paste("\n--- Checking and Installing", shinylive_pkg, "---"))

# First, ensure 'pak' is usable from the persistent library
if (!check_package_persistent("pak", persistent_lib_path)) {
   # This should have been handled above. If not, stop.
   stop("'pak' package is not correctly installed or loadable from the persistent library. Cannot install shinylive using pak.")
} else {
   print("'pak' package is available in the persistent library.")
   # Ensure pak is loaded for the pak::pak call if it wasn't automatically
   if (!"pak" %in% loadedNamespaces()) {
       # Load it specifically from the persistent library path
       library(pak, lib.loc = persistent_lib_path)
       print("Loaded 'pak' package.")
   }
}

# Now check for shinylive
cat("\nChecking:", shinylive_pkg, "...\n")
if (check_package_persistent(shinylive_pkg, persistent_lib_path)) {
  cat(shinylive_pkg, "is already available in the persistent library:", find.package(shinylive_pkg), "\n")
} else {
   # Reason for installation
    if (!requireNamespace(shinylive_pkg, quietly = TRUE)) {
       cat(shinylive_pkg, "is not available. Installing using pak to", persistent_lib_path, "...\n")
    } else {
       cat(shinylive_pkg, "is available but not in the persistent library (found at:", find.package(shinylive_pkg), ").\n")
       cat("Re-installing", shinylive_pkg, "using pak to persistent library:", persistent_lib_path, "...\n")
    }

  # Use pak::pak to install, specifying the library path
  tryCatch({
      pak::pak(shinylive_repo, lib = persistent_lib_path)

      # Verify installation attempt
      Sys.sleep(1) # Brief pause
      if (check_package_persistent(shinylive_pkg, persistent_lib_path)) {
          cat("Successfully installed and verified", shinylive_pkg, "in", persistent_lib_path, "using pak.\n")
      } else {
          warning(paste("Failed to install or verify", shinylive_pkg, "in", persistent_lib_path, "using pak."))
           if(shinylive_pkg %in% installed.packages(lib.loc = persistent_lib_path)[,"Package"] && !requireNamespace(shinylive_pkg, lib.loc=persistent_lib_path, quietly = TRUE)){
               cat("Package", shinylive_pkg, "files seem present but package is not loadable. Check for R version compatibility or installation errors.\n")
           }
      }
  }, error = function(e){
      warning(paste("Error during pak::pak installation of", shinylive_pkg, ":", e$message))
  })
}


# --- 5. Load packages ---
# Now that packages are hopefully installed correctly in the persistent lib, load them.
print("\n--- Loading Libraries ---")
tryCatch({
    library(shiny)
    library(shinylive) # Or other packages you need
    print("Libraries loaded successfully.")
}, error = function(e) {
    warning(paste("Error loading libraries:", e$message))
    print("Please check the installation status of the required packages in the persistent library.")
})


print("\n--- Setup Complete ---")
print(paste("Persistent library path:", persistent_lib_path))
print(paste("Using R version:", R.version.string))
print("Installed packages in persistent library:")
# Show only packages in the persistent library
print(installed.packages(lib.loc = persistent_lib_path)[, c("Package", "Version", "LibPath")])


SyntaxError: invalid syntax (983556949.py, line 16)

In [8]:
%%R
# --- Install Development Version of 'archive' Locally (Corrected Repo Path) ---

# 1. Install 'remotes' package if you don't have it
if (!requireNamespace("remotes", quietly = TRUE)) {
  cat("Installing 'remotes' package...\n")
  install.packages("remotes", repos = "https://cloud.r-project.org/")
}

# 2. Install 'archive' development version from GitHub using the CORRECT path
# The correct repository seems to be 'r-lib/archive'
correct_repo <- "r-lib/archive" # <-- CORRECTED PATH
cat("\nAttempting to install development version of 'archive' from GitHub repo:", correct_repo, "...\n")

tryCatch({
  # Use the corrected repository path here
  remotes::install_github(correct_repo)
}, error = function(e) {
  # Provide specific feedback if it fails again
  detailed_error <- conditionMessage(e)
  cat("Failed to install 'archive' from GitHub repo '", correct_repo, "':\n", detailed_error, "\n")
  if (grepl("HTTP error 404", detailed_error)) {
     cat("--> Still getting HTTP 404. Please double-check the repository URL manually on GitHub: https://github.com/", correct_repo, "\n")
  } else if (grepl("compilation failed|make error", detailed_error, ignore.case = TRUE)) {
     cat("--> Installation failed during compilation. Required build tools might be missing or code incompatible.\n")
  }
  stop("Installation from GitHub failed.")
})

# 3. Verify the installation
cat("\nVerifying installation of development 'archive'...\n")
load_success_dev <- FALSE
archive_version_dev <- NULL
tryCatch({
    # library() will load from the default path where remotes installed it
    library(archive)
    load_success_dev <- TRUE
    # Get version - it might look different from the CRAN version (e.g., 1.1.12.9000)
    archive_version_dev <- as.character(packageVersion("archive"))
}, error = function(e) {
    cat("--> Error loading development 'archive':", conditionMessage(e), "\n")
    # Specifically check for the 'undefined symbol' error again
    if (grepl("undefined symbol: R_new_custom_connection", conditionMessage(e))) {
         cat("--> NOTE: The 'undefined symbol' error persists, indicating incompatibility with R 4.5.0.\n")
    }
})

if(load_success_dev) {
    cat("--> SUCCESS: Successfully installed and loaded development 'archive'.\n")
    cat("    Version:", archive_version_dev, "\n")
    cat("    You can now try running your shinylive scripts again.\n")
} else {
    cat("--> FAILURE: Failed to load development 'archive'.\n")
    cat("    The incompatibility with R 4.5.0 might persist even in the dev version,\n")
    cat("    or there might be other build/load issues.\n")
    cat("    Using a stable R version is still the most recommended solution.\n")
}


Attempting to install development version of 'archive' from GitHub repo: r-lib/archive ...
── R CMD build ─────────────────────────────────────────────────────────────────
* checking for file ‘/tmp/RtmpcB61JL/remotes48c711ebd26/r-lib-archive-5e04bfa/DESCRIPTION’ ... OK
* preparing ‘archive’:
* checking DESCRIPTION meta-information ... OK
* cleaning src
* running ‘cleanup’
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
* building ‘archive_1.1.12.9000.tar.gz’


Verifying installation of development 'archive'...
--> Error loading development 'archive': package or namespace load failed for ‘archive’:
 .onLoad failed in loadNamespace() for 'archive', details:
  call: dyn.load(lib_path)
  error: unable to load shared object '/content/drive/MyDrive/R_Packages_Colab/R_4.5.0/archive/lib//libconnection.so':
  /content/drive/MyDrive/R_Packages_Colab/R_4.5.0/archive/lib//libconnection.so: undefined symbol: R_new_custom_connec

Installing package into ‘/content/drive/MyDrive/R_Packages_Colab/R_4.5.0’
(as ‘lib’ is unspecified)
In addition: Warning message:
In i.p(...) :
  installation of package ‘/tmp/RtmpcB61JL/file48c383672e9/archive_1.1.12.9000.tar.gz’ had non-zero exit status


In [19]:
# @title Setup Shinylive Assets on Google Drive (Run Once/Occasionally)

%%R
# ============================================================================
# Script 1: Setup Shinylive Assets on Google Drive (Run Once/Occasionally)
# ============================================================================
# Purpose: Downloads Shinylive web assets to a persistent Google Drive location.

# --- 1. Ensure Google Drive is Mounted ---
print("Ensure Google Drive is mounted at /content/drive")
# If not mounted, run in a Python cell:
# from google.colab import drive
# drive.mount('/content/drive')
Sys.sleep(2) # Brief pause to allow mounting messages

# --- 2. Define Persistent Shinylive Asset Path on Google Drive ---
# CHANGE 'MyDrive' if your main Google Drive folder name is different
drive_asset_path <- "/content/drive/MyDrive/Shinylive_Assets_Colab" # <<< CHOOSE YOUR DRIVE PATH

# --- 3. Create Asset Directory if Needed ---
if (!dir.exists(drive_asset_path)) {
  cat("Creating Shinylive asset directory on Google Drive:", drive_asset_path, "\n")
  dir.create(drive_asset_path, recursive = TRUE, showWarnings = FALSE)
  if(!dir.exists(drive_asset_path)){
      stop(paste("Failed to create Shinylive asset directory:", drive_asset_path, "- Check Drive permissions and path."))
  }
} else {
  cat("Using existing Shinylive asset directory on Google Drive:", drive_asset_path, "\n")
}

# --- 4. Set Shinylive Cache Directory Option ---
# This tells shinylive where to *store* the downloaded assets
options(shinylive.cache_dir = drive_asset_path)
cat("Set R option 'shinylive.cache_dir' to:", getOption("shinylive.cache_dir"), "\n")

# --- 5. Load Shinylive Package ---
# Assumes shinylive is installed (ideally in a persistent R library on Drive)
cat("\nLoading shinylive package...\n")
if (!requireNamespace("shinylive", quietly = TRUE)) {
    stop("shinylive package not found. Please install it first (preferably to a persistent library).")
}
library(shinylive)

# --- 6. Log Versions ---
cat("R version:", R.version.string, "\n")
cat("shinylive R package version:", as.character(packageVersion("shinylive")), "\n")

# --- 7. Download/Update Assets ---
# This is the core step: downloads assets to the specified Google Drive path.
# It should only download if assets are missing or if the required version changed.
cat("\nDownloading/Updating Shinylive assets in:", getOption("shinylive.cache_dir"), "...\n")
detach("package:archive")
library(archive)
tryCatch({
    shinylive::assets_download()
    cat("Asset download/update process completed.\n")
}, error = function(e) {
    stop("Failed to download/update shinylive assets to '", getOption("shinylive.cache_dir"), "': ", conditionMessage(e))
})

# --- 8. Verify Asset Information ---
cat("\nVerifying downloaded Shinylive assets:\n")
tryCatch({
    print(shinylive::assets_info())
}, error = function(e){
     cat("Error checking assets info after download:", conditionMessage(e), "\n")
     warning("Could not verify assets via assets_info(). Check the cache directory manually if needed.")
})

cat("\n--- Asset Setup Script Finished ---\n")
cat("Assets should now be available for use in:", getOption("shinylive.cache_dir"), "\n")

[1] "Ensure Google Drive is mounted at /content/drive"
Using existing Shinylive asset directory on Google Drive: /content/drive/MyDrive/Shinylive_Assets_Colab 
Set R option 'shinylive.cache_dir' to: /content/drive/MyDrive/Shinylive_Assets_Colab 

Loading shinylive package...
R version: R version 4.5.0 (2025-04-11) 
shinylive R package version: 0.3.0.9000 

Downloading/Updating Shinylive assets in: /content/drive/MyDrive/Shinylive_Assets_Colab ...
Error in detach("package:archive") : invalid 'name' argument


RInterpreterError: Failed to parse and evaluate line '# ============================================================================\n# Script 1: Setup Shinylive Assets on Google Drive (Run Once/Occasionally)\n# ============================================================================\n# Purpose: Downloads Shinylive web assets to a persistent Google Drive location.\n\n# --- 1. Ensure Google Drive is Mounted ---\nprint("Ensure Google Drive is mounted at /content/drive")\n# If not mounted, run in a Python cell:\n# from google.colab import drive\n# drive.mount(\'/content/drive\')\nSys.sleep(2) # Brief pause to allow mounting messages\n\n# --- 2. Define Persistent Shinylive Asset Path on Google Drive ---\n# CHANGE \'MyDrive\' if your main Google Drive folder name is different\ndrive_asset_path <- "/content/drive/MyDrive/Shinylive_Assets_Colab" # <<< CHOOSE YOUR DRIVE PATH\n\n# --- 3. Create Asset Directory if Needed ---\nif (!dir.exists(drive_asset_path)) {\n  cat("Creating Shinylive as

In [5]:
%%R
setwd("/content/longhaiSK/software")

# Define directories
input_dir <- "carkmark"
output_dir <- paste0(input_dir,"_shiny")

cat("Input_dir ", input_dir, "\n")

cat("Output_dir ", output_dir, "\n")
# Install required packages

# Load shinylive
library(shinylive)

# Log R and package versions
cat("R version:", R.version.string, "\n")
cat("shinylive version:", as.character(packageVersion("shinylive")), "\n")


# Display asset information
cat("Current Shinylive assets:\n")
print(shinylive::assets_info())


# Verify input directory
if (!dir.exists(input_dir)) {
  stop("Input directory '", input_dir, "' does not exist.")
}

# Export with warning and error capture
cat("Exporting Shinylive app from", input_dir, "to", output_dir, "...\n")


shinylive::export(
  appdir = input_dir,
  destdir = output_dir,
  prefer_cran = TRUE,  # Default repository
  webr_timeout = 180,  # 3-minute timeout
  verbose = TRUE       # Detailed output
)

#httpuv::runStaticServer(output_dir)




Input_dir  carkmark 
Output_dir  carkmark_shiny 
R version: R version 4.5.0 (2025-04-11) 
shinylive version: 0.3.0.9000 
Current Shinylive assets:
shinylive R package version: 0.3.0.9000
shinylive web assets version: 0.9.1

Local cached shinylive asset dir:
→ '/root/.cache/shinylive'

Installed assets:
• '(None)'
Error in x[[2]] : subscript out of bounds


RInterpreterError: Failed to parse and evaluate line 'setwd("/content/longhaiSK/software")\n\n# Define directories\ninput_dir <- "carkmark"\noutput_dir <- paste0(input_dir,"_shiny")\n\ncat("Input_dir ", input_dir, "\\n")\n\ncat("Output_dir ", output_dir, "\\n")\n# Install required packages\n\n# Load shinylive\nlibrary(shinylive)\n\n# Log R and package versions\ncat("R version:", R.version.string, "\\n")\ncat("shinylive version:", as.character(packageVersion("shinylive")), "\\n")\n\n# Clean up old assets\n# cat("Cleaning up old Shinylive assets...\\n")\n# #shinylive::assets_cleanup()\n#\n# # Update assets\n# cat("Downloading latest Shinylive assets...\\n")\n# shinylive::assets_download()\n\n# Display asset information\ncat("Current Shinylive assets:\\n")\nprint(shinylive::assets_info())\n\n\n# Verify input directory\nif (!dir.exists(input_dir)) {\n  stop("Input directory \'", input_dir, "\' does not exist.")\n}\n\n# Export with warning and error capture\ncat("Exporting Shinylive app fro

In [ ]:
# @title Mount google drive

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive
